# Video Game Sales Analysis - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the Video Game Sales dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [2]:
# Load dataset
df = pd.read_csv('../data/Video_Games.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (16719, 16)


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Sales Distribution Analysis

In [3]:
# Regional sales distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for i, col in enumerate(sales_cols):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel('Sales (millions)')
    ax.set_ylabel('Frequency')

# Remove empty subplot
fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.savefig('../outputs/figures/sales_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Top Platforms by Game Count

In [4]:
top_platforms = df['Platform'].value_counts().head(10)
plt.figure(figsize=(10, 6))
top_platforms.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 10 Platforms by Number of Games')
plt.xlabel('Platform')
plt.ylabel('Number of Games')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/top_platforms.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Top Publishers by Global Sales

In [5]:
top_publishers = df.groupby('Publisher')['Global_Sales'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
top_publishers.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Top 10 Publishers by Global Sales')
plt.xlabel('Publisher')
plt.ylabel('Global Sales (millions)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/figures/top_publishers.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Sales by Year of Release

In [6]:
sales_by_year = df.groupby('Year_of_Release')['Global_Sales'].sum().reset_index()
sales_by_year = sales_by_year[(sales_by_year['Year_of_Release'] >= 1996) & (sales_by_year['Year_of_Release'] <= 2016)]

plt.figure(figsize=(12, 6))
plt.bar(sales_by_year['Year_of_Release'], sales_by_year['Global_Sales'], color='mediumseagreen', edgecolor='black')
plt.title('Total Global Sales per Year (1996-2016)')
plt.xlabel('Year')
plt.ylabel('Global Sales (millions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/sales_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Sales by User Score and Critic Score

In [7]:
# Convert scores to numeric
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')
df['Critic_Score'] = pd.to_numeric(df['Critic_Score'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# User Score
user_sales = df.dropna(subset=['User_Score']).groupby('User_Score')['Global_Sales'].sum()
axes[0].bar(user_sales.index.astype(str), user_sales.values, color='skyblue', edgecolor='black')
axes[0].set_title('Global Sales by User Score')
axes[0].set_xlabel('User Score')
axes[0].set_ylabel('Global Sales (millions)')
axes[0].tick_params(axis='x', rotation=45)

# Critic Score
critic_sales = df.dropna(subset=['Critic_Score']).groupby('Critic_Score')['Global_Sales'].sum()
axes[1].bar(critic_sales.index.astype(str), critic_sales.values, color='lightcoral', edgecolor='black')
axes[1].set_title('Global Sales by Critic Score')
axes[1].set_xlabel('Critic Score')
axes[1].set_ylabel('Global Sales (millions)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/sales_by_scores.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Genre Analysis

In [8]:
# Genre frequency
genre_freq = df['Genre'].value_counts()
plt.figure(figsize=(10, 8))
plt.pie(genre_freq.values, labels=genre_freq.index, autopct='%1.1f%%', startangle=90)
plt.title('Genre Distribution')
plt.tight_layout()
plt.savefig('../outputs/figures/genre_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [9]:
# Regional sales by genre
region_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
genre_region = df.groupby('Genre')[region_cols].sum()
genre_region_melted = genre_region.reset_index().melt(id_vars='Genre', var_name='Region', value_name='Sales')

plt.figure(figsize=(14, 8))
sns.barplot(data=genre_region_melted, x='Genre', y='Sales', hue='Region')
plt.title('Regional Sales by Genre')
plt.xlabel('Genre')
plt.ylabel('Sales (millions)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Region')
plt.tight_layout()
plt.savefig('../outputs/figures/genre_regional_sales.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Multi-platform Analysis

In [10]:
# Create multi-platform indicator
game_counts = df['Name'].value_counts()
multiplatform_games = game_counts[game_counts > 1].index
df['Multiple_Platforms'] = df['Name'].apply(lambda x: 'yes' if x in multiplatform_games else 'no')

multiplat_sales = df[df['Multiple_Platforms'] == 'yes']['Global_Sales'].sum()
singleplat_sales = df[df['Multiple_Platforms'] == 'no']['Global_Sales'].sum()

plt.figure(figsize=(8, 6))
plt.bar(['Multi-platform', 'Single-platform'], [multiplat_sales, singleplat_sales], 
        color=['red', 'blue'], edgecolor='black')
plt.title('Global Sales: Multi-platform vs Single-platform Games')
plt.ylabel('Global Sales (millions)')
plt.tight_layout()
plt.savefig('../outputs/figures/multiplatform_sales.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Multi-platform sales: {multiplat_sales:.2f}M')
print(f'Single-platform sales: {singleplat_sales:.2f}M')
print(f'Multi-platform advantage: {((multiplat_sales - singleplat_sales) / singleplat_sales * 100):.1f}%')

Multi-platform sales: 4689.03M
Single-platform sales: 4231.27M
Multi-platform advantage: 10.8%


## 8. Correlation Matrix

In [11]:
# Correlation of regional sales
sales_corr = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(sales_corr, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: Regional Sales')
plt.tight_layout()
plt.savefig('../outputs/figures/sales_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Genre Sales by Year (Stacked)

In [12]:
genre_year = df.groupby(['Year_of_Release', 'Genre'])['Global_Sales'].sum().reset_index()
genre_year = genre_year[(genre_year['Year_of_Release'] >= 1996) & (genre_year['Year_of_Release'] <= 2016)]

genre_totals = genre_year.pivot(
    index='Year_of_Release', columns='Genre', values='Global_Sales'
).fillna(0)
plt.figure(figsize=(16, 8))
bottom = np.zeros(len(genre_totals))
for genre in genre_totals:
    sales = genre_totals[genre].to_numpy()
    plt.bar(genre_totals.index, sales, bottom=bottom, label=genre, alpha=0.8)
    bottom += sales

plt.title('Genre Sales by Year (Stacked)')
plt.xlabel('Year')
plt.ylabel('Global Sales (millions)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/figures/genre_sales_by_year.png', dpi=300, bbox_inches='tight')
plt.show()